# Llama 3.1 8B Instruct — QLoRA supervised fine-tuning (PEFT)

Trains on `finetuning_dataset.json` rows with `instruction`, `input`, `output` (ignores `_metadata`).

**Before you run**

1. **Hugging Face**: Request access to **Llama 3.1 8B Instruct** on the model card, then add your token:
   - **Kaggle**: Add a Secret `HF_TOKEN` (Settings → Secrets) and enable it for this notebook.
   - **Local**: `export HF_TOKEN=...` or run `huggingface-cli login`.
2. **GPU**: Prefer **GPU T4 x2** (or any ≥15 GB VRAM for 4-bit + LoRA; larger is safer).
3. **Data path**: Set `DATA_JSON` below to your uploaded `finetuning_dataset.json` (e.g. Kaggle Dataset mount path).

In [ ]:
# Optional: Kaggle — attach HF_TOKEN secret, then:
# import os; from kaggle_secrets import UserSecretsClient; os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

import os

if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") and not os.environ.get("HF_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient

        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        print("Could not load HF_TOKEN from Kaggle secrets:", e)

In [ ]:
%pip install -q "transformers>=4.43.0" "accelerate>=0.33.0" "peft>=0.12.0" "bitsandbytes>=0.43.0" "trl>=0.15.0" "datasets>=2.20.0" "sentencepiece" "protobuf"

In [ ]:
import json
import os
import random
from pathlib import Path

import torch
from datasets import Dataset
from huggingface_hub import login
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTTrainer

try:
    from trl import SFTConfig
except ImportError as e:
    raise ImportError("Install trl>=0.15 (see pip cell).") from e

# --- paths (edit for Kaggle) ---
# Kaggle example: DATA_JSON = "/kaggle/input/your-dataset-name/finetuning_dataset.json"
DATA_JSON = os.environ.get(
    "FINETUNE_DATA_JSON",
    str(Path("../data/jsons/finetuning_dataset.json").resolve()),
)
OUTPUT_DIR = os.environ.get("FINETUNE_OUTPUT_DIR", "./llama31-peca-lora")
MODEL_ID = os.environ.get("FINETUNE_MODEL_ID", "meta-llama/Meta-Llama-3.1-8B-Instruct")

HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
else:
    print("Warning: HF_TOKEN not set. Run huggingface-cli login or set the env var.")

EPOCHS = int(os.environ.get("FINETUNE_EPOCHS", "2"))
MAX_SEQ_LENGTH = int(os.environ.get("FINETUNE_MAX_SEQ_LENGTH", "4096"))
LORA_R = int(os.environ.get("FINETUNE_LORA_R", "16"))
LORA_ALPHA = int(os.environ.get("FINETUNE_LORA_ALPHA", "32"))

print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
path = Path(DATA_JSON)
if not path.is_file():
    raise FileNotFoundError(f"Dataset not found: {path.resolve()}")

with open(path, "r", encoding="utf-8") as f:
    raw = json.load(f)

rows = []
for row in raw:
    if not isinstance(row, dict):
        continue
    ins = row.get("instruction") or ""
    inp = row.get("input") or ""
    out = row.get("output") or ""
    if not str(out).strip():
        continue
    rows.append({"instruction": ins, "input": inp, "output": out})

random.seed(42)
random.shuffle(rows)
split = max(1, int(len(rows) * 0.95))
train_rows = rows[:split]
eval_rows = rows[split:]

train_ds = Dataset.from_list(train_rows)
eval_ds = Dataset.from_list(eval_rows) if eval_rows else None

print("Examples:", len(rows), "train:", len(train_rows), "eval:", len(eval_rows))

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
def formatting_prompts_func(example):
    messages = [
        {"role": "system", "content": example["instruction"]},
        {"role": "user", "content": example["input"]},
        {"role": "assistant", "content": example["output"]},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )


# Smoke test one formatted string
print(formatting_prompts_func(train_rows[0])[:1200], "...")

In [ ]:
import inspect

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

# Build kwargs compatible across TRL versions.
sft_sig = inspect.signature(SFTConfig.__init__).parameters
args_kwargs = dict(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=5,
    save_strategy="epoch",
    save_total_limit=2,
    bf16=use_bf16,
    fp16=use_fp16,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    report_to="none",
)

if eval_ds is not None:
    if "eval_strategy" in sft_sig:
        args_kwargs["eval_strategy"] = "steps"
    elif "evaluation_strategy" in sft_sig:
        args_kwargs["evaluation_strategy"] = "steps"
    args_kwargs["eval_steps"] = max(10, len(train_ds) // 10)
else:
    if "eval_strategy" in sft_sig:
        args_kwargs["eval_strategy"] = "no"
    elif "evaluation_strategy" in sft_sig:
        args_kwargs["evaluation_strategy"] = "no"

if "max_seq_length" in sft_sig:
    args_kwargs["max_seq_length"] = MAX_SEQ_LENGTH
elif "max_length" in sft_sig:
    args_kwargs["max_length"] = MAX_SEQ_LENGTH

args = SFTConfig(**args_kwargs)

_trainer_kw = dict(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    formatting_func=formatting_prompts_func,
)

if "processing_class" in inspect.signature(SFTTrainer.__init__).parameters:
    _trainer_kw["processing_class"] = tokenizer
else:
    _trainer_kw["tokenizer"] = tokenizer

trainer = SFTTrainer(**_trainer_kw)
trainer.train()

In [ ]:
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved adapter + tokenizer to:", OUTPUT_DIR)

## Load the adapter for a quick generation smoke test

**VRAM:** restart the kernel or run `del trainer; import gc; gc.collect(); torch.cuda.empty_cache()` before this cell so a second 4-bit base model fits.

In production, load `MODEL_ID` in 4-bit (or full precision) and attach the same LoRA path, using **identical** `apply_chat_template` as in training.

In [ ]:
import gc

import torch
from peft import PeftModel

try:
    del trainer
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
merged = PeftModel.from_pretrained(base, OUTPUT_DIR)
device = next(merged.parameters()).device

messages = [
    {"role": "system", "content": train_rows[0]["instruction"]},
    {"role": "user", "content": train_rows[0]["input"][:800]},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(device)
with torch.no_grad():
    out = merged.generate(**inputs, max_new_tokens=256, do_sample=True, temperature=0.7)
print(tokenizer.decode(out[0], skip_special_tokens=True)[-1200:])